# 🎬 Cinéfilo — SBC de regras para recomendar filmes

**Mini-Projeto 1 — Sistemas Baseados em Conhecimento (UFPB)**

Sistema Baseado em Conhecimento com regras IF–THEN em **experta**. O usuário descreve a sessão
(companhia, crianças, tempo, humor e época), o motor de inferência aplica **12 regras em 3 níveis de
encadeamento**, explica cada decisão e, ao final, a consulta montada pelas regras é enviada à
**API da TMDB** para buscar filmes reais.

```
Entrada (Sessao) ──► Nível 1 (R1–R6) ──► Nível 2 (R7–R9, R11–R12) ──► Nível 3 (R10) ──► Consulta ──► TMDB
```

### Como rodar
1. Crie uma conta gratuita em [themoviedb.org](https://www.themoviedb.org/) e gere uma chave em
   *Configurações → API* (serve a **API Key v3** ou o **token de leitura v4**).
2. No Colab, abra o ícone 🔑 (**Secrets**), crie o segredo `TMDB_API_KEY` com a chave e ative o acesso do notebook.
   Se preferir, o notebook pede a chave ao rodar.
3. `Ambiente de execução → Executar tudo`. A última célula pede os dados da sessão.

> Sem chave, o motor de regras, o trace, a explicação e os casos de teste funcionam normalmente; só a busca na TMDB é pulada.

## 1. Instalação e importações

In [ ]:
!pip install -q experta requests

In [ ]:
# Compatibilidade: o experta depende do frozendict 1.2, que usa collections.Mapping
# (removido no Python 3.10+). Este "shim" restaura os nomes antes do import.
import collections, collections.abc
for _nome in ("Mapping", "MutableMapping", "Sequence", "Iterable", "Callable"):
    if not hasattr(collections, _nome):
        setattr(collections, _nome, getattr(collections.abc, _nome))

from datetime import date
import requests
from experta import KnowledgeEngine, Fact, Rule, AS, MATCH, NOT, P, L
print("experta importado com sucesso")

## 2. Modelagem do domínio

Cada fato representa **um único conceito** do domínio; nenhum dado é guardado em dois fatos.

| Fato | Papel | Campos |
|---|---|---|
| `Sessao` | **Entrada** do usuário | `companhia`, `criancas`, `tempo`, `humor`, `epoca` |
| `Restricao` | Classificação indicativa máxima | `cert` (`L` ou `18`) |
| `Genero` | Gêneros a buscar (combinados com OU) | `ids` (IDs da TMDB) |
| `Periodo` | Faixa de lançamento | `inicio`, `fim` |
| `Criterio` | Como ordenar os resultados | `ordenar`, `votos_min` |
| `Duracao` | Duração máxima do filme | `max` |
| `Consulta` | **Saída** do motor | `params` (prontos para `/discover/movie`) |

Todo fato derivado leva também o campo `regras`: a cadeia de regras que o produziu. É isso que permite
responder *"porque R1, R3 e R7 dispararam"* para cada decisão.

In [ ]:
# Tabelas do domínio (IDs oficiais de gêneros da TMDB)
GENEROS_TMDB = {
    28: "Ação", 12: "Aventura", 16: "Animação", 35: "Comédia", 80: "Crime",
    18: "Drama", 10751: "Família", 14: "Fantasia", 27: "Terror", 9648: "Mistério",
    10749: "Romance", 878: "Ficção científica", 53: "Suspense", 10752: "Guerra",
}

# Usado pela R3: humor desejado -> gêneros base
GENEROS_POR_HUMOR = {
    "rir":        [35],          # Comédia
    "emocionar":  [18],          # Drama
    "adrenalina": [28, 53],      # Ação, Suspense
    "medo":       [27],          # Terror
    "pensar":     [878, 9648],   # Ficção científica, Mistério
}

# Usado pela R7: gêneros inadequados para crianças e seus substitutos
PROIBIDOS_INFANTIL = {27, 53, 80, 10752}
SUBSTITUTOS_INFANTIS = [16, 10751, 12]   # Animação, Família, Aventura

DURACAO_MAX_INFANTIL = 100   # minutos (R8)
DURACAO_MIN_PADRAO = 60      # minutos (R8)
FOLGA_DURACAO = 30           # minutos: largura mínima da faixa de duração (R8)
VOTOS_MIN_CLASSICO = 200     # votos (R11)
SESSAO_LONGA = 150           # minutos (R12)
DURACAO_MIN_LONGA = 90       # minutos (R12)
ROMANCE = 10749              # (R9)

OPCOES = {
    "companhia": ["sozinho", "casal", "amigos", "familia"],
    "humor": list(GENEROS_POR_HUMOR),
    "epoca": ["recente", "classico", "tanto_faz"],
}

def nomes_generos(ids):
    return ", ".join(GENEROS_TMDB.get(i, str(i)) for i in ids)

# ---------------- Fatos ----------------
class Sessao(Fact):
    """ENTRADA. companhia, criancas (bool), tempo (min), humor, epoca."""
class Restricao(Fact):
    """Classificação indicativa máxima: cert ('L' ou '18')."""
class Genero(Fact):
    """Lista de IDs de gênero TMDB a buscar (combinados com OU)."""
class Periodo(Fact):
    """Faixa de lançamento: inicio / fim (AAAA-MM-DD ou None)."""
class Criterio(Fact):
    """Critério de ordenação: ordenar (sort_by TMDB) e votos_min."""
class Duracao(Fact):
    """Faixa de duração aceitável, em minutos: min / max."""
class Consulta(Fact):
    """SAÍDA do motor: parâmetros prontos para /discover/movie da TMDB."""

# Todo fato derivado carrega o campo `regras`: a cadeia de regras que o produziu.
# É isso que permite explicar cada decisão ("porque R1 e R7 dispararam").

## 3. Base de regras

| Regra | Nível | Salience | SE | ENTÃO |
|---|---|---|---|---|
| **R1** | 1 | 10 | há crianças na sessão | classificação máxima **L** |
| **R2** | 1 | −10 | **não** existe Restricao (padrão) | sessão adulta, **sem filtro** de classificação |
| **R3** | 1 | 0 | humor = H | gêneros base conforme o humor |
| **R4** | 1 | 0 | época = E | faixa de anos (recente: últimos 5 anos; clássico: até 1999) |
| **R5** | 1 | 10 | companhia = amigos ou família | ordenar por **popularidade** |
| **R6** | 1 | −10 | **não** existe Criterio (padrão) | ordenar pelos **mais bem avaliados** |
| **R7** | 2 | 5 | classificação L **e** gêneros impróprios (terror, suspense, crime, guerra) | troca por Animação, Família, Aventura |
| **R8** | 2 | 0 | tempo disponível T **e** classificação C | duração de 60 min até T (T limitado a 100 se C = L; o piso recua se a faixa ficar estreita) |
| **R9** | 2 | 5 | casal **e** classificação 18 **e** humor rir/emocionar **e** sem Romance | adiciona **Romance** |
| **R11** | 2 | 5 | época = clássico **e** mínimo de votos acima de 200 | reduz o mínimo para **200 votos** |
| **R12** | 2 | 5 | duração máxima ≥ 150 min **e** piso ainda em 60 | eleva o piso para **90 min** |
| **R10** | 3 | −50 | Restricao, Genero, Duracao, Periodo e Criterio existem **e não** existe Consulta | monta a **consulta à TMDB** |

### Encadeamento (3 níveis)
- **Nível 1** lê só a `Sessao` e cria `Restricao`, `Genero`, `Periodo` e `Criterio`.
- **Nível 2** lê fatos do nível 1: `R8` depende de `Restricao` para criar `Duracao`; `R7` e `R9` ajustam `Genero`;
  `R11` ajusta `Criterio` à luz de `Periodo`; `R12` ajusta `Duracao` criada pela `R8`.
- **Nível 3** (`R10`) depende de `Duracao`, que só existe depois do nível 2. Por isso **toda** execução percorre os 3 níveis.

### Resolução de conflito (salience + NOT)
- **R1 × R2** e **R5 × R6**: as regras padrão (R2, R6) têm salience −10 e condição `NOT(...)`. Quando uma
  regra específica (R1 ou R5, salience 10) dispara antes, ela cria o fato e a ativação da regra padrão é
  **cancelada** — nunca existem duas classificações nem dois critérios.
- **R7/R9 × R10**: os ajustes do nível 2 têm salience maior que R10 (−50), garantindo que a consulta só é
  montada com os gêneros já corrigidos.

### Ausência de loops
- R7 só dispara se houver gênero impróprio, e o resultado não tem nenhum → não reativa.
- R9 só dispara se Romance **não** estiver na lista, e o resultado tem Romance → não reativa.
- R11 exige mínimo de votos acima de 200 e grava exatamente 200 → não reativa.
- R12 exige piso abaixo de 90 e grava exatamente 90 → não reativa.
- R10 exige `NOT(Consulta())` → dispara uma única vez.

In [ ]:
REGRAS = {
    "R1":  (1, "Se há crianças na sessão, a classificação máxima é Livre (L)."),
    "R2":  (1, "Se não há restrição infantil, a sessão é adulta e não recebe filtro de classificação (regra padrão)."),
    "R3":  (1, "O humor desejado define os gêneros base."),
    "R4":  (1, "A época preferida define a faixa de anos de lançamento."),
    "R5":  (1, "Sessões em grupo (amigos ou família) priorizam filmes populares."),
    "R6":  (1, "Se nenhum critério foi definido, priorizam-se os filmes mais bem avaliados (regra padrão)."),
    "R7":  (2, "Se a classificação é L e há gêneros impróprios, eles são trocados por gêneros infantis."),
    "R8":  (2, "A duração máxima é o tempo disponível, limitada a 100 min quando a classificação é L."),
    "R9":  (2, "Casal adulto querendo rir ou se emocionar recebe Romance entre os gêneros."),
    "R11": (2, "Clássicos exigem menos votos, porque acumulam menos avaliações na TMDB."),
    "R12": (2, "Sessão longa pede filme substancial: a duração mínima sobe de 60 para 90 min."),
    "R10": (3, "Com classificação, gêneros, duração, período e critério definidos, monta-se a consulta à TMDB."),
}

def _ordem(rid):
    return int(rid[1:])


class Cinefilo(KnowledgeEngine):
    """Sistema baseado em conhecimento para recomendar um filme para uma sessão."""

    def __init__(self):
        super().__init__()
        self.trace = []

    # ---------- infraestrutura de trace / explicação ----------
    def _disparou(self, rid, detalhe, *cadeias):
        """Registra o disparo no trace e devolve a cadeia de justificativa."""
        cadeia = {rid}
        for c in cadeias:
            cadeia |= set(c)
        cadeia = tuple(sorted(cadeia, key=_ordem))
        nivel, _ = REGRAS[rid]
        self.trace.append({"passo": len(self.trace) + 1, "regra": rid,
                           "nivel": nivel, "detalhe": detalhe, "cadeia": cadeia})
        return cadeia

    # ================= NÍVEL 1: Sessao -> fatos básicos =================

    @Rule(Sessao(criancas=True), salience=10)
    def r1_restricao_infantil(self):
        r = self._disparou("R1", "Há crianças → classificação máxima L.")
        self.declare(Restricao(cert="L", regras=r))

    # Resolução de conflito: R2 é a regra padrão (salience baixa + NOT).
    # Se R1 disparar antes, o fato Restricao passa a existir e a ativação de R2 é cancelada.
    @Rule(Sessao(), NOT(Restricao()), salience=-10)
    def r2_restricao_padrao(self):
        r = self._disparou("R2", "Sem crianças → sessão adulta, sem filtro de classificação.")
        self.declare(Restricao(cert="18", regras=r))

    @Rule(Sessao(humor=MATCH.humor))
    def r3_genero_por_humor(self, humor):
        ids = GENEROS_POR_HUMOR[humor]
        r = self._disparou("R3", f"Humor '{humor}' → gêneros base: {nomes_generos(ids)}.")
        self.declare(Genero(ids=ids, regras=r))

    @Rule(Sessao(epoca=MATCH.epoca))
    def r4_periodo(self, epoca):
        if epoca == "recente":
            ini, fim = f"{date.today().year - 5}-01-01", None
            txt = f"lançados a partir de {date.today().year - 5}"
        elif epoca == "classico":
            ini, fim = None, "1999-12-31"
            txt = "lançados até 1999"
        else:
            ini, fim = None, None
            txt = "qualquer ano"
        r = self._disparou("R4", f"Época '{epoca}' → filmes {txt}.")
        self.declare(Periodo(inicio=ini, fim=fim, regras=r))

    @Rule(Sessao(companhia=L("amigos") | L("familia")), salience=10)
    def r5_criterio_grupo(self):
        r = self._disparou("R5", "Sessão em grupo → ordenar por popularidade.")
        self.declare(Criterio(ordenar="popularity.desc", votos_min=500, regras=r))

    # Resolução de conflito: mesma estratégia de R2 (padrão com salience baixa + NOT).
    @Rule(Sessao(), NOT(Criterio()), salience=-10)
    def r6_criterio_padrao(self):
        r = self._disparou("R6", "Nenhum critério definido → ordenar pelos mais bem avaliados.")
        self.declare(Criterio(ordenar="vote_average.desc", votos_min=1000, regras=r))

    # ================= NÍVEL 2: ajustes sobre fatos do nível 1 =================

    @Rule(Restricao(cert="L", regras=MATCH.rr),
          AS.g << Genero(ids=P(lambda ids: bool(set(ids) & PROIBIDOS_INFANTIL)),
                         regras=MATCH.rg),
          salience=5)
    def r7_genero_seguro(self, g, rr, rg):
        antigos = list(g["ids"])
        novos = [i for i in antigos if i not in PROIBIDOS_INFANTIL]
        novos += [i for i in SUBSTITUTOS_INFANTIS if i not in novos]
        r = self._disparou("R7", f"Classificação L proíbe {nomes_generos(set(antigos) & PROIBIDOS_INFANTIL)}"
                                 f" → gêneros passam a ser {nomes_generos(novos)}.", rr, rg)
        self.modify(g, ids=novos, regras=r)   # novos não têm proibidos: R7 não dispara de novo

    @Rule(Sessao(tempo=MATCH.tempo), Restricao(cert=MATCH.cert, regras=MATCH.rr))
    def r8_duracao(self, tempo, cert, rr):
        if cert == "L" and tempo > DURACAO_MAX_INFANTIL:
            maximo = DURACAO_MAX_INFANTIL
            txt = f"{tempo} min disponíveis, mas com classificação L o limite é {maximo} min."
        else:
            maximo = tempo
            txt = f"duração máxima = tempo disponível ({tempo} min)."
        # O piso não pode encostar no teto: uma faixa de largura zero não devolve nada.
        minimo = min(DURACAO_MIN_PADRAO, maximo - FOLGA_DURACAO)
        if minimo < DURACAO_MIN_PADRAO:
            txt += f" Piso recuado para {minimo} min para a faixa não ficar vazia."
        r = self._disparou("R8", txt[0].upper() + txt[1:], rr)
        self.declare(Duracao(min=minimo, max=maximo, regras=r))

    @Rule(Sessao(companhia="casal", humor=L("rir") | L("emocionar")),
          Restricao(cert="18", regras=MATCH.rr),
          AS.g << Genero(ids=P(lambda ids: ROMANCE not in ids), regras=MATCH.rg),
          salience=5)
    def r9_romance_casal(self, g, rr, rg):
        novos = list(g["ids"]) + [ROMANCE]
        r = self._disparou("R9", f"Casal adulto → Romance adicionado: {nomes_generos(novos)}.", rr, rg)
        self.modify(g, ids=novos, regras=r)   # agora contém Romance: R9 não dispara de novo

    # Clássicos da TMDB acumulam menos votos que lançamentos recentes; manter o mínimo
    # alto descartaria justamente os filmes que a R4 pediu.
    @Rule(Periodo(fim=P(lambda f: f is not None), regras=MATCH.rp),
          AS.c << Criterio(votos_min=P(lambda v: v > VOTOS_MIN_CLASSICO), regras=MATCH.rc),
          salience=5)
    def r11_votos_classico(self, c, rp, rc):
        antes = c["votos_min"]
        r = self._disparou("R11", f"Busca por clássicos → mínimo de votos cai de {antes} "
                                  f"para {VOTOS_MIN_CLASSICO}.", rp, rc)
        self.modify(c, votos_min=VOTOS_MIN_CLASSICO, regras=r)   # já no piso: R11 não dispara de novo

    @Rule(AS.d << Duracao(min=P(lambda m: m < DURACAO_MIN_LONGA),
                          max=P(lambda m: m >= SESSAO_LONGA), regras=MATCH.rd),
          salience=5)
    def r12_duracao_minima(self, d, rd):
        r = self._disparou("R12", f"Sessão longa ({d['max']} min disponíveis) → filme com pelo "
                                  f"menos {DURACAO_MIN_LONGA} min.", rd)
        self.modify(d, min=DURACAO_MIN_LONGA, regras=r)   # min já é 90: R12 não dispara de novo

    # ================= NÍVEL 3: conclusão =================

    # salience mínima: só dispara depois de todos os ajustes do nível 2.
    @Rule(Restricao(cert=MATCH.cert, regras=MATCH.ra),
          Genero(ids=MATCH.ids, regras=MATCH.rb),
          Duracao(min=MATCH.dmin, max=MATCH.dmax, regras=MATCH.rc),
          Periodo(inicio=MATCH.ini, fim=MATCH.fim, regras=MATCH.rd),
          Criterio(ordenar=MATCH.ordem, votos_min=MATCH.votos, regras=MATCH.re),
          NOT(Consulta()),
          salience=-50)
    def r10_montar_consulta(self, cert, ids, dmin, dmax, ini, fim, ordem, votos, ra, rb, rc, rd, re):
        params = {
            "language": "pt-BR",
            "include_adult": "false",
            "with_genres": "|".join(str(i) for i in ids),   # "|" = OU na TMDB
            "with_runtime.gte": dmin,
            "with_runtime.lte": dmax,
            "sort_by": ordem,
            "vote_count.gte": votos,
        }
        if cert == "L":
            # certification.lte = "no máximo L"; certification seria "exatamente L"
            params.update({"certification_country": "BR", "certification.lte": "L"})
        if ini:
            params["primary_release_date.gte"] = ini
        if fim:
            params["primary_release_date.lte"] = fim
        r = self._disparou("R10", "Todos os critérios definidos → consulta TMDB montada.",
                           ra, rb, rc, rd, re)
        self.declare(Consulta(params=params, regras=r))

## 4. Trace e explicação

- `mostrar_trace` lista cada disparo em ordem, com nível e o que a regra concluiu, e quais regras não dispararam.
- `explicar` justifica **cada decisão** com a cadeia de regras que a produziu.
- `mostrar_agenda` exibe o conjunto de conflito antes da execução, tornando a estratégia de resolução visível.

In [ ]:
def _porque(cadeia):
    ids = list(cadeia)
    lista = ids[0] if len(ids) == 1 else ", ".join(ids[:-1]) + " e " + ids[-1]
    verbo = "disparou" if len(ids) == 1 else "dispararam"
    return f"porque {lista} {verbo}"

def _fato(engine, classe):
    for f in engine.facts.values():
        if isinstance(f, classe):
            return f
    return None

def mostrar_trace(engine):
    print("TRACE (ordem de disparo)")
    print("-" * 78)
    for t in engine.trace:
        print(f"{t['passo']:>2}. [{t['regra']:<3}| nível {t['nivel']}] {t['detalhe']}")
    nao = [r for r in REGRAS if r not in {t['regra'] for t in engine.trace}]
    print("-" * 78)
    print("Regras que NÃO dispararam:", ", ".join(nao) if nao else "nenhuma")

def explicar(engine):
    """Explicação textual de cada decisão, com a cadeia de regras que a justifica."""
    res, gen = _fato(engine, Restricao), _fato(engine, Genero)
    dur, per = _fato(engine, Duracao), _fato(engine, Periodo)
    cri, con = _fato(engine, Criterio), _fato(engine, Consulta)
    ordem = {"popularity.desc": "mais populares", "vote_average.desc": "mais bem avaliados"}
    if per["inicio"]:
        periodo = f"a partir de {per['inicio'][:4]}"
    elif per["fim"]:
        periodo = f"até {per['fim'][:4]}"
    else:
        periodo = "qualquer ano"
    print("\nEXPLICAÇÃO DAS DECISÕES")
    print("-" * 78)
    cert_txt = ("Classificação máxima L" if res["cert"] == "L"
                else "Sessão adulta, sem filtro de classificação")
    print(f"• {cert_txt}, {_porque(res['regras'])}.")
    print(f"• Gêneros: {nomes_generos(gen['ids'])}, {_porque(gen['regras'])}.")
    print(f"• Duração entre {dur['min']} e {dur['max']} min, {_porque(dur['regras'])}.")
    print(f"• Lançamento: {periodo}, {_porque(per['regras'])}.")
    print(f"• Ordenação: {ordem[cri['ordenar']]}, {_porque(cri['regras'])}.")
    print(f"• Consulta final à TMDB, {_porque(con['regras'])}.")

def inferir(companhia, criancas, tempo, humor, epoca, verbose=True):
    """Roda o motor para uma sessão e devolve (engine, params da consulta)."""
    engine = Cinefilo()
    engine.reset()
    engine.declare(Sessao(companhia=companhia, criancas=criancas,
                          tempo=tempo, humor=humor, epoca=epoca))
    engine.run()
    params = dict(_fato(engine, Consulta)["params"])
    if verbose:
        mostrar_trace(engine)
        explicar(engine)
    return engine, params

def regras_disparadas(engine):
    return {t["regra"] for t in engine.trace}


def mostrar_agenda(engine):
    """Mostra o conjunto de conflito (agenda) antes de rodar o motor."""
    engine.strategy.update_agenda(engine.agenda, *engine.get_activations())
    print("AGENDA INICIAL (conjunto de conflito — a de maior salience dispara primeiro)")
    ids = set()
    # o experta guarda a agenda em ordem crescente e dispara a partir do fim
    for i, act in enumerate(reversed(engine.agenda.activations), 1):
        nome = act.rule._wrapped.__name__
        rid = nome.split("_")[0].upper()
        ids.add(rid)
        print(f"  {i}. {rid:<3} salience {act.rule.salience:>4}  {REGRAS[rid][1]}")
    return ids

### Demonstração da resolução de conflito
Com crianças e família, **R1 × R2** e **R5 × R6** entram juntas na agenda. As de maior salience disparam
primeiro e cancelam as regras padrão.

In [ ]:
demo = Cinefilo()
demo.reset()
demo.declare(Sessao(companhia="familia", criancas=True, tempo=120, humor="medo", epoca="tanto_faz"))
na_agenda = mostrar_agenda(demo)
demo.run()
print()
mostrar_trace(demo)
canceladas = sorted(na_agenda - regras_disparadas(demo), key=_ordem)
print("\nEstavam na agenda mas foram CANCELADAS pela resolução de conflito:", ", ".join(canceladas))

## 5. Integração com a TMDB e entrada do usuário

A consulta que o motor monta (fato `Consulta`, criado pela R10) é enviada ao endpoint
[`/discover/movie`](https://developer.themoviedb.org/reference/discover-movie). Para cada filme retornado,
o sistema explica quais regras o justificam.

In [ ]:
import os, html
from IPython.display import display, HTML

TMDB_URL = "https://api.themoviedb.org/3/discover/movie"
DETALHE_URL = "https://api.themoviedb.org/3/movie/{}"
POSTER_URL = "https://image.tmdb.org/t/p/w185"

def obter_chave():
    """Procura a chave nos Secrets do Colab, depois em variável de ambiente, depois pergunta."""
    try:
        from google.colab import userdata
        chave = userdata.get("TMDB_API_KEY")
        if chave:
            return chave.strip()
    except Exception:
        pass
    if os.environ.get("TMDB_API_KEY"):
        return os.environ["TMDB_API_KEY"].strip()
    from getpass import getpass
    return getpass("Chave da TMDB (API Key v3 ou token v4; Enter para pular): ").strip()

TMDB_CHAVE = obter_chave()
print("Chave TMDB carregada." if TMDB_CHAVE else "Sem chave: o motor roda, mas a busca na TMDB será pulada.")

def _autenticar(params=None):
    """Monta params e headers conforme o tipo de chave (v3 ou v4)."""
    params, headers = dict(params or {}), {"accept": "application/json"}
    if TMDB_CHAVE.startswith("eyJ"):            # token de leitura v4
        headers["Authorization"] = f"Bearer {TMDB_CHAVE}"
    else:                                        # API Key v3
        params["api_key"] = TMDB_CHAVE
    return params, headers

def buscar_duracao(filme_id):
    """A /discover/movie nao devolve runtime; ele so existe em /movie/{id}."""
    if not TMDB_CHAVE:
        return None
    params, headers = _autenticar()
    try:
        resp = requests.get(DETALHE_URL.format(filme_id), params=params,
                            headers=headers, timeout=15)
    except requests.RequestException:
        return None
    if resp.status_code != 200:
        return None
    return resp.json().get("runtime") or None   # 0 ou ausente = duracao desconhecida

def buscar_filmes(params, n=5):
    """Envia a consulta montada pela R10 ao endpoint /discover/movie da TMDB."""
    if not TMDB_CHAVE:
        print("\n[TMDB] Sem chave configurada — consulta não enviada.")
        return []
    params, headers = _autenticar(params)
    try:
        resp = requests.get(TMDB_URL, params=params, headers=headers, timeout=15)
    except requests.RequestException as erro:
        print(f"\n[TMDB] Falha de conexão: {type(erro).__name__}")
        return []
    if resp.status_code != 200:                  # não imprime a URL para não vazar a chave
        dica = " (chave inválida?)" if resp.status_code == 401 else ""
        print(f"\n[TMDB] Erro HTTP {resp.status_code}{dica}")
        return []
    return resp.json().get("results", [])[:n]

def mostrar_filmes(filmes, engine):
    """Exibe os filmes e explica, para cada um, quais regras o justificam."""
    if not filmes:
        dur, cri = _fato(engine, Duracao), _fato(engine, Criterio)
        print("Nenhum filme retornado pela TMDB para esses critérios.")
        print(f"   Os filtros mais estreitos são a duração ({dur['min']}–{dur['max']} min) "
              f"e o mínimo de {cri['votos_min']} votos.")
        print("   Informar mais tempo disponível é o que costuma alargar mais a busca.")
        return
    gen, res, dur = _fato(engine, Genero), _fato(engine, Restricao), _fato(engine, Duracao)
    cartoes = []
    for i, f in enumerate(filmes):
        comuns = [g for g in f.get("genre_ids", []) if g in gen["ids"]]
        minutos = buscar_duracao(f["id"])        # uma chamada extra por filme
        if not minutos:                          # a TMDB nem sempre tem o dado
            faixa = f"até {dur['max']} min"
        elif dur["min"] <= minutos <= dur["max"]:
            faixa = f"{minutos} min, dentro dos {dur['min']}–{dur['max']} permitidos"
        else:                                    # não deveria ocorrer: a TMDB filtra no servidor
            faixa = f"{minutos} min, fora dos {dur['min']}–{dur['max']} pedidos"
        motivo = (f"Gêneros {nomes_generos(comuns)} ({', '.join(gen['regras'])}); "
                  f"{faixa} ({', '.join(dur['regras'])}); "
                  f"classificação {res['cert']} ({', '.join(res['regras'])}).")
        ano = (f.get("release_date") or "????")[:4]
        dur_txt = f" — {minutos} min" if minutos else ""
        titulo = "⭐ RECOMENDAÇÃO PRINCIPAL" if i == 0 else f"Alternativa {i}"
        print(f"\n{titulo}: {f['title']} ({ano}){dur_txt} — nota {f.get('vote_average', 0):.1f}")
        print(f"   Por quê: {motivo}")
        poster = (f'<img src="{POSTER_URL}{f["poster_path"]}" style="width:92px;border-radius:6px">'
                  if f.get("poster_path") else "")
        sinopse = html.escape((f.get("overview") or "Sem sinopse em português.")[:280])
        cartoes.append(
            f'<div style="display:flex;gap:12px;margin:8px 0;padding:8px;border:1px solid #ccc;border-radius:8px">'
            f'{poster}<div><b>{titulo}: {html.escape(f["title"])} ({ano})</b>{dur_txt} — ⭐ {f.get("vote_average", 0):.1f}'
            f'<br><small>{sinopse}</small><br><small><i>Por quê: {html.escape(motivo)}</i></small></div></div>')
    display(HTML("".join(cartoes)))

def recomendar(companhia, criancas, tempo, humor, epoca, n=5):
    """Pipeline completo: entrada → motor de regras → trace/explicação → TMDB."""
    engine, params = inferir(companhia, criancas, tempo, humor, epoca)
    print("\nParâmetros enviados à TMDB:", params)
    mostrar_filmes(buscar_filmes(params, n), engine)
    return engine, params

In [ ]:
def _escolher(pergunta, opcoes):
    print(pergunta)
    for i, o in enumerate(opcoes, 1):
        print(f"  {i}. {o}")
    while True:
        r = input("> ").strip()
        if r.isdigit() and 1 <= int(r) <= len(opcoes):
            escolha = opcoes[int(r) - 1]
            print(f"  → {escolha}")   # eco: a caixa de input nao aparece na saida salva
            return escolha
        print("Opção inválida: digite o número de uma das opções acima.")

def coletar_entrada():
    """Coleta os dados da sessão com validação."""
    companhia = _escolher("Com quem você vai assistir?", OPCOES["companhia"])
    criancas = _escolher("Há crianças na sessão?", ["sim", "nao"]) == "sim"
    print("Quanto tempo vocês têm? (em minutos, de 60 a 300)")
    while True:
        t = input("> ").strip()
        if t.isdigit() and 60 <= int(t) <= 300:
            tempo = int(t)
            print(f"  → {tempo} min")
            break
        print("Valor inválido: digite um número inteiro entre 60 e 300.")
    humor = _escolher("Qual o clima da sessão?", OPCOES["humor"])
    epoca = _escolher("Preferência de época?", OPCOES["epoca"])
    return dict(companhia=companhia, criancas=criancas, tempo=tempo, humor=humor, epoca=epoca)


## 6. Casos de teste

Cada caso tem a saída esperada comentada e é verificado com `assert` sobre as **regras disparadas** e a
**consulta montada** (que são determinísticas). Os filmes exibidos vêm da TMDB em tempo real e podem variar.

In [ ]:
# CASO 1 — Família com crianças querendo "medo", 120 min, qualquer época
# Saída esperada:
#   • R1 dispara (há crianças) → classificação L. R2 é cancelada pelo NOT(Restricao).
#   • R5 dispara (família) → ordenar por popularidade. R6 é cancelada pelo NOT(Criterio).
#   • R3 → Terror; em seguida R7 troca Terror por Animação, Família e Aventura.
#   • R8 → limita a 100 min (sessão infantil), apesar dos 120 disponíveis.
#   • R9 não se aplica (não é casal).
#   • R11 e R12 não se aplicam (época indiferente; sessão não é longa).
#   • Consulta: with_genres=16|10751|12, certification.lte=L, duração 60–100, popularity.desc.
caso1 = dict(companhia="familia", criancas=True, tempo=120, humor="medo", epoca="tanto_faz")
engine1, params1 = recomendar(**caso1)

assert regras_disparadas(engine1) == {"R1", "R3", "R4", "R5", "R7", "R8", "R10"}
assert params1["with_genres"] == "16|10751|12"
assert params1["certification.lte"] == "L" and params1["certification_country"] == "BR"
assert params1["with_runtime.gte"] == 60 and params1["with_runtime.lte"] == 100
assert params1["sort_by"] == "popularity.desc"
print("\n✅ Caso 1: saída conforme o esperado")

In [ ]:
# CASO 2 — Casal sem crianças querendo se emocionar, 150 min, filmes recentes
# Saída esperada:
#   • R1 não dispara → R2 (padrão) define classificação 18.
#   • R3 → Drama; R9 adiciona Romance (casal adulto + emocionar).
#   • R4 → lançados nos últimos 5 anos.
#   • R5 não se aplica (casal) → R6 (padrão) ordena pelos mais bem avaliados.
#   • R8 → duração até 150 min; R12 eleva o piso para 90 min (sessão longa).
#   • R7 não se aplica (sem restrição infantil); R11 também não (época recente).
#   • Consulta: with_genres=18|10749, sem filtro de classificação, duração 90–150.
caso2 = dict(companhia="casal", criancas=False, tempo=150, humor="emocionar", epoca="recente")
engine2, params2 = recomendar(**caso2)

assert regras_disparadas(engine2) == {"R2", "R3", "R4", "R6", "R8", "R9", "R12", "R10"}
assert params2["with_genres"] == "18|10749"
assert "certification.lte" not in params2
assert params2["with_runtime.gte"] == 90 and params2["with_runtime.lte"] == 150
assert params2["primary_release_date.gte"] == f"{date.today().year - 5}-01-01"
assert params2["sort_by"] == "vote_average.desc"
print("\n✅ Caso 2: saída conforme o esperado")

In [ ]:
# CASO 3 — Sozinho, querendo adrenalina, 110 min, clássicos
# Saída esperada:
#   • R2 (padrão) → classificação 18; R6 (padrão) → mais bem avaliados.
#   • R3 → Ação e Suspense, mantidos (R7 não se aplica: não há crianças).
#   • R4 → lançados até 1999; R11 baixa o mínimo de votos de 1000 para 200.
#   • R8 → duração até 110 min. R12 não se aplica (sessão não é longa).
#   • R9 não se aplica (sozinho, humor adrenalina).
#   • Consulta: with_genres=28|53, primary_release_date.lte=1999-12-31, vote_count.gte=200.
caso3 = dict(companhia="sozinho", criancas=False, tempo=110, humor="adrenalina", epoca="classico")
engine3, params3 = recomendar(**caso3)

assert regras_disparadas(engine3) == {"R2", "R3", "R4", "R6", "R8", "R11", "R10"}
assert params3["with_genres"] == "28|53"
assert params3["primary_release_date.lte"] == "1999-12-31"
assert params3["with_runtime.lte"] == 110
assert params3["vote_count.gte"] == 200
assert params3["sort_by"] == "vote_average.desc"
print("\n✅ Caso 3: saída conforme o esperado")

In [ ]:
# Verificações globais da base de regras
from collections import Counter
engines = [engine1, engine2, engine3]

# 1) Sem loops: nenhuma regra dispara mais de uma vez por execução
for i, e in enumerate(engines, 1):
    repetidas = [r for r, n in Counter(t["regra"] for t in e.trace).items() if n > 1]
    assert not repetidas, f"Caso {i}: regras repetidas {repetidas}"

# 2) Cobertura: todas as regras disparam em pelo menos um dos casos
cobertas = set().union(*(regras_disparadas(e) for e in engines))
assert cobertas == set(REGRAS), f"Não cobertas: {set(REGRAS) - cobertas}"

# 3) Encadeamento: toda execução passa pelos 3 níveis
for e in engines:
    assert {t["nivel"] for t in e.trace} == {1, 2, 3}

print(f"✅ {len(REGRAS)} regras, todas exercitadas nos testes, sem loops e com 3 níveis em todas as execuções.")

## 7. Use você mesmo

Responda às perguntas e o sistema recomenda um filme, explicando o raciocínio.

In [ ]:
entrada = coletar_entrada()
print()
_ = recomendar(**entrada)